# Wave, wind and tide climate at a point — seasonal tables and XBeach boundary conditions

Give a point offshore of your site and get:

- **waves** (height, peak and mean period, direction, swell and wind-sea parts) and **wind**, as statistics per
  month and per season, direction tables and roses, Hs × direction and Hs × Tp tables;
- **tide** (constituents, tidal levels HAT … LAT, spring / neap ranges) and **storm surge** per season;
- **representative cases** per season (typical, energetic, storm) and ready-made **XBeach boundary files** for each;
- one Excel workbook with every table, CSV files and figures.

| Data | Source | Coverage |
|---|---|---|
| waves (Hs, Tp, Tm, direction, partitions) | Copernicus Marine wave reanalysis + analysis (global 0.2°/1/12°, 3-hourly; or Mediterranean 1/24°, hourly) | 1980 (Med 1985) → now |
| waves (Hs, Tm, direction), 10 m wind | ERA5, Climate Data Store point time-series service | 1940 → ~6 days ago, hourly |
| tide, storm surge | GTSM v3 reanalysis (Deltares), Climate Data Store | 1950 → 2024 |

No Earth Engine. **Everything lives in one folder of your Google Drive**: download
[Wave_climate.zip](https://github.com/Alerovere/seastrong-field-tools/releases/download/wave-climate-latest/Wave_climate.zip) (latest version), unzip it, upload the `Wave_climate` folder anywhere in your Drive and open
this notebook from there (double-click → *Open with Google Colaboratory*). Results go to its `outputs` subfolder.

Every step has an **explanation** cell, an **options** cell (every option with its default and a comment; a misspelt
option name stops with a hint) and a **run** cell. Downloads are cached: running again only fetches what is new.

## Before you start: accounts and Colab Secrets

Free accounts at the **Copernicus Climate Data Store** (accept the licences of *ERA5 hourly time-series data on single
levels* and *Global sea level change time series from 1950 to 2050…* on the dataset pages, *Download* tab) and
**Copernicus Marine**. Store the log-in details as **Colab Secrets** (key icon on the left → *Add new secret*, switch
on *Notebook access*). They are never written into the notebook or the outputs.

| Secret | Value |
|---|---|
| `CDS_KEY` | Climate Data Store personal access token (your CDS profile page) |
| `CMEMS_USERNAME`, `CMEMS_PASSWORD` | Copernicus Marine log-in |

## Install (from your Drive folder)

Set `TOOLKIT_DIR` to the `Wave_climate` folder in your Drive. The cell mounts Drive, installs the packages it needs
(with a *constraints file* so that Colab's own Google packages are not replaced) and imports the code from that
folder. Nothing is downloaded from GitHub unless you set `UPDATE_FROM_GITHUB = True` (code files replaced by the latest
release, the old `src` folder backed up, `outputs` and this notebook untouched; then *Runtime → Restart session*).

In [ ]:
TOOLKIT_DIR = '/content/drive/MyDrive/SEASTRONG/Wave_climate'   # the Wave_climate folder you saved in Drive
UPDATE_FROM_GITHUB = False    # True = replace the code by the latest released version (outputs are kept)

import subprocess, sys, re, os, glob
from google.colab import drive
drive.mount('/content/drive')
cands = [TOOLKIT_DIR, os.path.join(TOOLKIT_DIR, 'Wave_climate')] + sorted(glob.glob(os.path.join(TOOLKIT_DIR, '*', 'Wave_climate')))
found = [c for c in cands if os.path.exists(os.path.join(c, 'pyproject.toml')) and os.path.isdir(os.path.join(c, 'src', 'wave_climate'))]
if not found:
    raise FileNotFoundError(f'No Wave_climate folder (with pyproject.toml and src/wave_climate) in {TOOLKIT_DIR}. '
                            'In the Files panel on the left, right-click the folder -> Copy path.')
TOOLKIT_DIR = found[0]; print('toolkit folder:', TOOLKIT_DIR)

if UPDATE_FROM_GITHUB:
    import urllib.request, zipfile, shutil, io, time
    url = 'https://github.com/Alerovere/seastrong-field-tools/releases/download/wave-climate-latest/Wave_climate.zip'
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
    pre = [n for n in z.namelist() if n.endswith('pyproject.toml')][0][:-len('pyproject.toml')]
    backup = os.path.join(TOOLKIT_DIR, f"_backup_src_{time.strftime('%Y%m%d_%H%M%S')}")
    shutil.copytree(os.path.join(TOOLKIT_DIR, 'src'), backup)
    for n in z.namelist():
        rel = n[len(pre):]
        if not rel or n.endswith('/') or rel.startswith(('outputs/', '_backup')): continue
        if rel.startswith('notebooks/') and rel.endswith('.ipynb'): rel = rel[:-6] + '_NEW.ipynb'
        dst = os.path.join(TOOLKIT_DIR, rel); os.makedirs(os.path.dirname(dst), exist_ok=True)
        with open(dst, 'wb') as f: f.write(z.read(n))
    print(f'Code updated (old code in {backup}). Runtime -> Restart session, set UPDATE_FROM_GITHUB = False, run again.')

frozen = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout.splitlines()
protect = re.compile(r'^(google-|googleapis-|grpcio|protobuf|proto-plus|httplib2|ipykernel|ipython|jupyter|notebook|tornado|requests|urllib3)', re.I)
with open('/content/colab_constraints.txt', 'w') as f:
    f.write('\n'.join(l for l in frozen if '==' in l and protect.match(l)))
import tomllib
deps = tomllib.load(open(os.path.join(TOOLKIT_DIR, 'pyproject.toml'), 'rb'))['project']['dependencies']
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *deps, '-c', '/content/colab_constraints.txt'], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-2500:])
if r.returncode != 0: raise RuntimeError('Installation failed (see above).')
sys.path.insert(0, os.path.join(TOOLKIT_DIR, 'src'))
OUTPUTS = os.path.join(TOOLKIT_DIR, 'outputs'); os.makedirs(OUTPUTS, exist_ok=True)
print('outputs folder:', OUTPUTS)

In [ ]:
import wave_climate as wc
import tomllib
print('wave_climate', wc.__version__, 'from', os.path.dirname(wc.__file__))
_v = tomllib.load(open(os.path.join(TOOLKIT_DIR, 'pyproject.toml'), 'rb'))['project']['version']
if wc.__version__ != _v or not wc.__file__.startswith(TOOLKIT_DIR):
    raise RuntimeError(f'Version {wc.__version__} is loaded but {TOOLKIT_DIR} has {_v}: Runtime -> Restart session and run again.')
# after editing the code in src/wave_climate: wc.reload(), then run the cells from the site cell down

## Site

- **`POINT`**: longitude, latitude **offshore**, in water deep enough for the models (beyond reefs, islands and
  headlands that shelter the coast), ideally near where your XBeach grid starts or where a wave transformation model
  (e.g. SWAN) would take over. The models describe open-sea conditions: inside a lagoon or behind a reef they are not
  valid. The download step reports how far the model cells are from the point.
- **`REGION`**: Copernicus Marine wave model, `'global'` or `'med'` (Mediterranean, finer, from 1985).
- **`NAME`**: start of every file name and the output subfolder.

In [ ]:
SITE = dict(
    NAME='Belize_south',          # file names and output subfolder
    POINT=(-88.30, 16.10),        # lon, lat offshore
    REGION='global',              # 'global' or 'med'
    OUTPUT_DIR=OUTPUTS,           # results: the 'outputs' subfolder of the toolkit folder
)
ctx = wc.setup(SITE)

## Step 1 — Download waves and wind

**What it does.** Downloads the full records at the point:
- **Copernicus Marine** waves: `hs` (significant height), `tp` (peak period), `tm10` (energy period Tm-1,0), `tm02`,
  `dir` (mean direction, coming from), wind-sea (`hs_windsea`, `tm_windsea`, `dir_windsea`) and primary swell
  (`hs_swell`, `tm_swell`, `dir_swell`). The reanalysis (to ~2 months ago) and the analysis (recent) are joined;
  forecasts are dropped. Every 3 hours (global) or hourly (Med).
- **ERA5**: `hs`, `tm` (ERA5 mean wave period), `dir`, and the 10 m wind `wind_speed`, `wind_from`, hourly. The ERA5
  point service has **no peak period**, so the XBeach cases use Copernicus Marine waves and ERA5 wind.

**Options.** `START`: a long record gives robust statistics and rare storms (30+ years recommended). The rest rarely
needs changing. First download: 5–20 minutes; afterwards it is cached.

In [ ]:
DOWNLOAD = dict(
    START='1980-01-01',           # first day (Copernicus Marine global starts 1980, Med 1985)
    END=None,                     # last day (None = latest available)
    ERA5=True,                    # ERA5 waves + wind
    CMEMS=True,                   # Copernicus Marine waves (needed for Tp and the XBeach cases)
    CMEMS_VARIABLES={'VHM0': 'hs', 'VTPK': 'tp', 'VTM10': 'tm10', 'VTM02': 'tm02', 'VMDR': 'dir',
                     'VHM0_WW': 'hs_windsea', 'VTM01_WW': 'tm_windsea', 'VMDR_WW': 'dir_windsea',
                     'VHM0_SW1': 'hs_swell', 'VTM01_SW1': 'tm_swell', 'VMDR_SW1': 'dir_swell'},
    CMEMS_SEARCH_DEG=None,        # box searched for the nearest sea cell (None = region default)
    ERA5_SEARCH_DEG=0.75,         # box searched for the nearest ERA5 wave cell (<= 1)
    ERA5_LAG_DAYS=6,              # ERA5 ends ~5 days before today
    ERA5_TS_CHUNK_YEARS=20,       # years per ERA5 request (split automatically if refused)
    CDS_RETRIES=3,
)
wc.download.run(ctx, DOWNLOAD)

## Step 2 — Wave and wind climate per month and season

**Seasons.** Monthly tables are always made. `SEASONS` adds your own groups of months (any names; a month may belong
to more than one). Use seasons that mean something for the site, e.g. for **Belize**: *Northers* Nov–Feb (cold fronts,
northerly winds and waves), *Dry* Mar–May (steady trades), *Wet / hurricane* Jun–Nov. The default is DJF / MAM / JJA / SON.

**Tables** (CSV in `tables/` and sheets of the Excel workbook; `period` = all year, each season, each month):
- `wave_statistics`: number of values, mean, std, percentiles (`PERCENTILES`) and max of Hs, Tp, Tm, partitions;
  circular mean direction and its variability (`dir_spread_deg`, how much the direction changes, not the spectral
  spreading of the waves);
- `wind_statistics`: same for ERA5 wind speed and direction (hourly, full record);
- `hs_x_direction`, `wind_speed_x_direction`: % of time in each height / speed class and direction sector;
- `hs_x_tp`: % of time in each Hs and Tp class;
- `era5_vs_cmems`: how the two wave models compare (a model comparison, not a validation).

**Figures**: monthly box plots, wave and wind roses per season, Hs–Tp per season.

**Options.** `WAVE_SOURCE='cmems'` (has Tp) is needed for the XBeach cases; `YEARS` restricts the statistics to a
period (e.g. the last 30 years); the bins set the classes of the tables and roses.

In [ ]:
CLIMATE = dict(
    SEASONS={'DJF': [12, 1, 2], 'MAM': [3, 4, 5], 'JJA': [6, 7, 8], 'SON': [9, 10, 11]},   # name: months
    WAVE_SOURCE='cmems',          # 'cmems' (with Tp; needed for XBeach) or 'era5'
    YEARS=None,                   # [first, last] year for the statistics (None = all downloaded)
    PERCENTILES=[10, 25, 50, 75, 90, 95, 99],
    DIR_SECTORS=16,               # direction sectors (16 = 22.5 deg)
    HS_BINS=[0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 6.0, 20.0],   # m
    TP_BINS=[0, 4, 6, 8, 10, 12, 14, 16, 18, 22, 30],                         # s
    WIND_BINS=[0, 2, 4, 6, 8, 10, 12, 15, 20, 40],                            # m/s
)
wc.climate.run(ctx, CLIMATE)

## Step 3 — Tide and storm surge (GTSM)

**What it does.**
1. Downloads one year of hourly **tidal elevation** from GTSM (`TIDE_YEARS`) and runs a **harmonic analysis**
   (UTide): amplitude and phase of each constituent (M2, S2, K1, O1, …).
2. Predicts the tide over `PREDICT_YEARS` (19, the nodal cycle) and computes the **tidal levels**: HAT, MHWS, MHHW,
   MHW, MHWN, MSL, MLWN, MLW, MLLW, MLWS, LAT, spring and neap ranges, and the **tide type** (form factor
   F = (K1+O1)/(M2+S2): semidiurnal < 0.25 < mixed < 3 < diurnal).
3. Downloads the **daily maximum storm surge** (`SURGE_YEARS`) and gives its percentiles per month and season and the
   annual maxima.

**Good to know.**
- GTSM files are **global** (the service cannot cut an area): each request downloads the whole world, the point
  nearest to your site is kept and the big files are deleted. Expect several minutes per year requested; the
  extracted point series are cached. Default: 1 year of tide + 30 years of surge.
- Levels are relative to the **GTSM model mean sea level (1986–2005)**, not chart datum or your survey datum: add the
  local offset before combining them with a DEM or bathymetry.
- GTSM has tide and wind/pressure surge but no waves (set-up), no river discharge and no seasonal steric sea-level
  cycle. Its tide near complex coasts (lagoons, reef channels) is approximate: compare with a local tide gauge if one
  exists.

In [ ]:
TIDES = dict(
    TIDES=True,                   # False = no tide / surge (the XBeach cases then use TIDE_CYCLE='none')
    TIDE_YEARS=[2023, 2023],      # years of tidal elevation for the harmonic analysis (1 year is enough)
    TIDE_AGGREGATION='hourly',    # 'hourly' or '10_min' (10-min files are 6x larger)
    SURGE_YEARS=[1995, 2024],     # years of daily-maximum surge (GTSM reanalysis ends 2024)
    GTSM_VERSION='v3',
    MAX_STATION_KM=30,            # the GTSM point must be within this distance of POINT
    PREDICT_YEARS=19,             # years of predicted tide for HAT / LAT
    KEEP_GLOBAL_FILES=False,      # keep the global GTSM files (several GB) instead of deleting them
)
wc.tides.run(ctx, TIDES, seasons=CLIMATE['SEASONS'], percentiles=CLIMATE['PERCENTILES'])

## Step 4 — Representative cases and XBeach boundary files

For each season and each level of `LEVELS` (e.g. typical = Hs P50, energetic = P90, storm = P99):
- **Hm0** = that percentile of the season's Hs;
- **Tp**, **mainang** (wave direction) and **wind** = median Tp, mean direction and median wind of the records whose Hs
  is within ±`BAND` of Hm0, so the combination actually occurs together (the table also gives Tp P10–P90);
- **water level** = a representative tidal day (`TIDE_CYCLE`: spring, mean or neap) + the season's daily-maximum surge
  at the same percentile (`SURGE_PERCENTILE`). Using the same percentile for waves and surge is cautious: big waves and
  high surge often, but not always, come together.

Files in `xbeach/<case>/`: `jonswap.txt` (`wbctype = jonstable`: Hm0 Tp mainang gammajsp s duration dtbc),
`wind.txt` (time, speed, direction), `tide.txt` (time, zs0), and `params_boundary.txt` with the matching params.txt
lines. All tables are collected in `<NAME>_wave_wind_tide_climate.xlsx`.

**Choose with care.**
- `SPREADING_S`: XBeach directional spreading (cosine power). Not available from these data sources: ~5–10 for
  wind sea, 20–30 or more for long swell; the `swell_share_of_energy` column of the cases helps decide.
- `GAMMA`: JONSWAP peak enhancement, 3.3 standard; lower (1–2) for broad, developing seas.
- Directions are **nautical, coming from**. Check `thetanaut` and the grid rotation `alfa` in your XBeach setup.
- The waves are offshore model values: if your XBeach boundary is in shallow water, transform them first (e.g. SWAN)
  or place the boundary offshore.

In [ ]:
CASES = dict(
    LEVELS={'typical': 50, 'energetic': 90, 'storm': 99},   # case name: Hs percentile within each season
    BAND=0.10,                    # records with Hs within +-10 % of the case Hs give Tp, direction and wind
    MIN_RECORDS=30,               # the band widens until this many records are found
    SURGE_PERCENTILE='same',      # 'same' as Hs, a number (e.g. 50), or 0 (no surge)
    TIDE_CYCLE='spring',          # 'spring', 'mean', 'neap' or 'none'
    DURATION_H=25,                # length of each case (h)
    GAMMA=3.3,                    # JONSWAP peak enhancement
    SPREADING_S=10,               # XBeach directional spreading s
    DTBC=1.0,                     # XBeach boundary time step (s)
    TIDE_DT_S=600,                # time step of tide.txt (s)
)
wc.cases.run(ctx, CASES)

## What to check before using the results

- `outputs/<NAME>/sources.json`: the model cells and GTSM point actually used and their distance from `POINT`.
- `era5_vs_cmems`: if the two models differ a lot, the point may be too close to the coast for one of them.
- The tidal levels against a tide gauge or published tide tables, if any exist for the site.
- Rare events: P99 of 30+ years is still not a design storm. Hurricanes are poorly resolved in ERA5 and the wave
  reanalyses (too coarse): for extreme-event design use dedicated hurricane datasets.